In [2]:
from PIL import Image
import requests
import re
import os
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities

In [22]:

def read_images_for_tile(tile_number, directory):
    images = []
    for i in range(6):
        # Construct the filename based on the tile number and image sequence
        filename = f"neg_rgb_{i}_tile_{tile_number}.jpg"
        filepath = os.path.join(directory, filename)
        
        # Check if the file exists and load the image
        if os.path.exists(filepath):
            image = Image.open(filepath)
            images.append(image)
        else:
            print(f"File {filename} does not exist.")
    
    return images

# Specify the tile number and directory containing images
tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

def get_unique_tiles(directory):
    tile_numbers = set()
    # Regular expression to match the tile number in the filename
    pattern = r'_(\d+)\.jpg'  # Matches the tile number before .jpg

    for filename in os.listdir(directory):
        match = re.search(pattern, filename)
        if match:
            tile_number = match.group(1)
            tile_numbers.add(tile_number)

    return sorted(tile_numbers)

In [23]:
tiles = get_unique_tiles(directory)

In [21]:
inputs = processor(text=["an aerial photo of a location with an ephemeral gully formed", "an aerial photo of a location with no ephemeral gully formed"], images=images, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1)  # we can take the softmax to get the label probabilities
print(probs)
# Determine if any image has a high probability for the gully class
gully_detected = (probs[:, 0] > 0.5).any().item()  # Check if any image has gully class probability > 0.5

if gully_detected:
    print("At least one image has an ephemeral gully.")
else:
    print("No ephemeral gully detected in any of the images.")

tile_number = 100
directory = '/root/home/data_jpg/'  # Replace with the path to your images

# Read images for the specified tile number
images = read_images_for_tile(tile_number, directory)

tensor([[0.2854, 0.7146],
        [0.3014, 0.6986],
        [0.1831, 0.8169],
        [0.1297, 0.8703],
        [0.2861, 0.7139],
        [0.1992, 0.8008]], grad_fn=<SoftmaxBackward0>)
No ephemeral gully detected in any of the images.


In [19]:
probs

tensor([[0.8889, 0.1111],
        [0.9538, 0.0462],
        [0.8749, 0.1251],
        [0.7142, 0.2858],
        [0.9312, 0.0688],
        [0.8492, 0.1508]], grad_fn=<SoftmaxBackward0>)

In [13]:
images

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=128x128>]